In [22]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys

import plotly.express as px

In [23]:
# Read distance matrix
dist_mat = pd.read_csv("../output/handshape_distance_matrix.csv")

dist_mat.head()

,Name,fbx\M20241107_6263_260710_2.fbx_segment_0,fbx\M20241107_6263_260710_2.fbx_segment_1,fbx\M20241107_6263_260710_2.fbx_segment_2,fbx\M20241107_6263_260710_2.fbx_segment_3,fbx\M20250916_4405_260709_0.fbx_segment_0,fbx\M20260119_1078_260713_1.fbx_segment_0,fbx\M20260119_1078_260713_1.fbx_segment_1,fbx\M20260119_1078_260713_1.fbx_segment_2,fbx\M20260119_1078_260713_1.fbx_segment_3,...,../HandformSession/HandformsTrimmed/Q.FBX,../HandformSession/HandformsTrimmed/R.FBX,../HandformSession/HandformsTrimmed/S.FBX,../HandformSession/HandformsTrimmed/Shh.FBX,../HandformSession/HandformsTrimmed/T.FBX,../HandformSession/HandformsTrimmed/T_Open.FBX,../HandformSession/HandformsTrimmed/V.FBX,../HandformSession/HandformsTrimmed/V_Curved.FBX,../HandformSession/HandformsTrimmed/W.FBX,../HandformSession/HandformsTrimmed/Y.FBX
0,fbx\M20241107_6263_260710_2.fbx_segment_0,0.000000,0.006577,0.037838,0.008648,0.008725,0.011049,0.031654,0.028377,0.023939,...,0.055018,0.043958,0.085497,0.025913,0.037260,0.040193,0.033086,0.054068,0.034298,0.038793
1,fbx\M20241107_6263_260710_2.fbx_segment_1,0.006577,0.000000,0.035703,0.002726,0.002730,0.003514,0.031243,0.028355,0.025071,...,0.061775,0.041847,0.085882,0.018139,0.028083,0.031941,0.030679,0.053242,0.037703,0.037691
2,fbx\M20241107_6263_260710_2.fbx_segment_2,0.037838,0.035703,0.000000,0.028830,0.026907,0.033182,0.002618,0.003054,0.008837,...,0.013591,0.037042,0.015801,0.081256,0.070314,0.086397,0.030318,0.025767,0.051231,0.018985
3,fbx\M20241107_6263_260710_2.fbx_segment_3,0.008648,0.002726,0.028830,0.000000,0.000261,0.000879,0.025623,0.023142,0.019196,...,0.049673,0.027282,0.071577,0.016042,0.019963,0.024357,0.021833,0.037620,0.033457,0.033581
4,fbx\M20250916_4405_260709_0.fbx_segment_0,0.008725,0.002730,0.026907,0.000261,0.000000,0.000639,0.024495,0.022112,0.019508,...,0.047286,0.027796,0.067524,0.017401,0.019494,0.024674,0.022410,0.036319,0.033076,0.031033


In [24]:
# Top 10 most similar handshapes to 'fbx\M20241107_6263_260710_2.fbx_segment_2'

# Get the row for the specified handshape
# Get file name from index ?
index = 10

file_name = dist_mat.index[index]
handshape_row = dist_mat.iloc[index]

# Get the indices of the 10 smallest distances
handshape_distances = handshape_row[1:].to_numpy()
similar_indices = handshape_distances.argsort()

# Get the names of the 10 most similar handshapes (excluding any names starting with 'fbx\')
similar_handshape_names = dist_mat.columns[similar_indices + 1]  # +1 to account for the first column being the name
similar_handshape_names = [name for name in similar_handshape_names if not name.startswith('fbx\\')]

print("10 most similar handshapes to:")
for handshape in similar_handshape_names:
    print(handshape)

10 most similar handshapes to:
../HandformSession/HandformsTrimmed/N.FBX
../HandformSession/HandformsTrimmed/V.FBX
../HandformSession/HandformsTrimmed/MiddleJitter.FBX
../HandformSession/HandformsTrimmed/Beak2_Open.FBX
../HandformSession/HandformsTrimmed/Middle.FBX
../HandformSession/HandformsTrimmed/1_Curved_Jitter-3.FBX
../HandformSession/HandformsTrimmed/1_1_Jitter.FBX
../HandformSession/HandformsTrimmed/Beak2_Open_Spread.FBX
../HandformSession/HandformsTrimmed/V_Curved.FBX
../HandformSession/HandformsTrimmed/M.FBX
../HandformSession/HandformsTrimmed/Beak.FBX
../HandformSession/HandformsTrimmed/1_2-3.FBX
../HandformSession/HandformsTrimmed/1_2-2.FBX
../HandformSession/HandformsTrimmed/O.FBX
../HandformSession/HandformsTrimmed/R.FBX
../HandformSession/HandformsTrimmed/G.FBX
../HandformSession/HandformsTrimmed/L2.FBX
../HandformSession/HandformsTrimmed/K.FBX
../HandformSession/HandformsTrimmed/D.FBX
../HandformSession/HandformsTrimmed/D-2.FBX
../HandformSession/HandformsTrimmed/Baby_B

In [25]:
# Get distance array
dist_mat = dist_mat.drop(columns=['Name'])
distance_array = dist_mat.to_numpy()

In [26]:
# # Make all distances 0 or higher
# distance_array += np.abs(np.min(distance_array))

In [27]:
# t-SNE on the distance matrix
random_num = random.randint(0, 10000)
# random_num = 8079  # This is the last one I used, so for reproducibility hardcoded
print("Using random seed:", random_num)
tsne_dist = manifold.TSNE(n_components=2, perplexity=50, learning_rate=200, init='random', metric='precomputed', method='barnes_hut', random_state=random_num)
X_tsne_dist = tsne_dist.fit_transform(distance_array)

Using random seed: 6045


In [28]:
# Dataframe for the px plot
tsne_df = pd.DataFrame({
    'tsne_1': X_tsne_dist[:, 0],
    'tsne_2': X_tsne_dist[:, 1],
    'name': dist_mat.columns,
    'is_handshape': [name.startswith("../") for name in dist_mat.columns]
})

# Plot
fig = px.scatter(
    tsne_df,
    x='tsne_1',
    y='tsne_2',
    color='is_handshape',
    hover_data=['name'],
    title="t-SNE Dimensionality Reduction of the Shape Features"
)

fig.update_layout(width=800, height=800)
fig.show()